In [ ]:
# Insert input video path
input_path = "sample_video.mp4" 

# Optional output path. This will change the title of the output video. Default is set to traffic_density_analysis_roi_cropped.mp4
output_path = None 

In [ ]:
# Install Ultralytics library
!pip install ultralytics > /dev/null 2>&1;

In [ ]:
# Disable warnings in the notebook to maintain clean output cells
import warnings
warnings.filterwarnings('ignore')

# Import necessary libraries
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import yaml
from PIL import Image
from ultralytics import YOLO
from IPython.display import Video

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# Construct the path to the best model weights file using os.path.join
best_model_path = 'best.pt'


# Load the best model weights into the YOLO model
best_model = YOLO(best_model_path)

## After Cropping Inference (Traffic Video 2)

In [ ]:
import cv2
import numpy as np

# Define the polygonal ROI
vertices = np.array([(1500, 200), (2250, 200), (3000, 650), (1000, 650)], dtype=np.int32)

# Threshold for considering traffic as heavy
heavy_traffic_threshold = 10

# Annotation settings
text_position = (100, 300)
intensity_position = (100, 100)
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 3
font_color = (255, 255, 255)
background_color = (0, 0, 255)

# Open the video
cap = cv2.VideoCapture(input_path)

# Define codec and output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
if output_path is None:
    out = cv2.VideoWriter('traffic_density_analysis_roi_cropped.mp4', fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))
else:
    out = cv2.VideoWriter(output_path, fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))

# Get bounding rectangle for polygonal ROI
x, y, w, h = cv2.boundingRect(vertices)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Extract ROI from the bounding rectangle
    roi_rect = frame[y:y+h, x:x+w]

    # Create mask same size as the ROI rectangle
    mask = np.zeros((h, w), dtype=np.uint8)

    # Shift vertices to rectangle coordinates
    shifted_vertices = vertices - [x, y]

    # Fill polygon on mask
    cv2.fillPoly(mask, [shifted_vertices], 255)

    # Apply mask to the cropped ROI
    roi_masked = cv2.bitwise_and(roi_rect, roi_rect, mask=mask)

    # Run YOLO inference on masked/cropped ROI
    results = best_model.predict(roi_masked, imgsz=640, conf=0.15)
    processed_roi = results[0].plot(line_width=1)

    # Count vehicles within polygon
    bounding_boxes = results[0].boxes
    vehicle_count = 0

    for box in bounding_boxes.xyxy:
        # Adjust box center coordinates to full-frame coordinates
        x_center = int((box[0] + box[2]) / 2) + x
        y_center = int((box[1] + box[3]) / 2) + y

        # Check if the center lies within the original polygon
        if cv2.pointPolygonTest(vertices, (x_center, y_center), False) >= 0:
            vehicle_count += 1

    # Prepare the final processed frame
    processed_frame = frame.copy()
    # Paste the processed ROI back into the original frame
    mask_rgb = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    inv_mask = cv2.bitwise_not(mask_rgb)
    roi_rect_clean = cv2.bitwise_and(roi_rect, inv_mask)
    roi_combined = cv2.add(roi_rect_clean, processed_roi)
    processed_frame[y:y+h, x:x+w] = roi_combined

    # Draw the polygon
    cv2.polylines(processed_frame, [vertices], isClosed=True, color=(0, 255, 0), thickness=2)

    # Determine traffic intensity
    traffic_intensity = "Heavy" if vehicle_count > heavy_traffic_threshold else "Smooth"

    # Annotate text
    # --- Vehicle Count Text & Background ---
    vehicle_text = f'Vehicle Count in ROI: {vehicle_count}'
    (text_width, text_height), _ = cv2.getTextSize(vehicle_text, font, font_scale, 2)
    top_left_vehicle = (text_position[0] - 10, text_position[1] - text_height - 10)
    bottom_right_vehicle = (text_position[0] + text_width + 10, text_position[1] + 10)
    cv2.rectangle(processed_frame, top_left_vehicle, bottom_right_vehicle, background_color, -1)
    cv2.putText(processed_frame, vehicle_text, text_position, font, font_scale, font_color, 2, cv2.LINE_AA)

    # --- Traffic Intensity Text & Background (with vertical spacing) ---
    vertical_spacing = 40  # Adjust this for more/less space between the boxes
    intensity_y_offset = text_position[1] + text_height + vertical_spacing

    intensity_text = f'Traffic Intensity: {traffic_intensity}'
    (text_width2, text_height2), _ = cv2.getTextSize(intensity_text, font, font_scale, 2)
    top_left_intensity = (text_position[0] - 10, intensity_y_offset - text_height2 - 10)
    bottom_right_intensity = (text_position[0] + text_width2 + 10, intensity_y_offset + 10)
    cv2.rectangle(processed_frame, top_left_intensity, bottom_right_intensity, background_color, -1)
    cv2.putText(processed_frame, intensity_text, (text_position[0], intensity_y_offset),
                font, font_scale, font_color, 2, cv2.LINE_AA)

    # Write output frame
    out.write(processed_frame)

    # Optional real-time view
    # cv2.imshow('ROI Analysis', processed_frame)
    # if cv2.waitKey(1) & 0xFF == ord('q'):
    #     break

cap.release()
out.release()
# cv2.destroyAllWindows()
